In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Locate the repo root without importing from src yet.
_current = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (_current, *_current.parents)
    if (candidate / "AGENTS.md").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_selection.data_loading import load_split, baseline_mean_metrics


In [2]:
X_train, y_train = load_split("train", processed_dir=PROJECT_ROOT / "data" / "processed")
X_val, y_val = load_split("validation", processed_dir=PROJECT_ROOT / "data" / "processed")

feature_cols = list(X_train.columns)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)


Train: (1895, 25)
Validation: (600, 25)


## 1. Fit once on train, rank all 25 features by gain-based importance

In [3]:
XGB_PARAMS = dict(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror",
)

xgb_model = XGBRegressor(**XGB_PARAMS)
xgb_model.fit(X_train, y_train)

importance_df = (
    pd.DataFrame({
        "feature": feature_cols,
        "importance": xgb_model.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

importance_df


,feature,importance
0,sma_60,0.068208
1,atr_14,0.056487
2,sma_5,0.056146
3,roc_20,0.055738
4,volatility_20,0.048901
5,price_to_sma_20,0.047480
6,macd_hist,0.045464
7,return_20d,0.045398
8,macd,0.045292
9,sma_20,0.044889


## 2. Top-K subsets vs. the train-mean baseline

In [4]:
top_k_list = [5, 10, 15, 20]

results = [baseline_mean_metrics(y_train, y_val)]

for k in top_k_list:
    top_features = importance_df["feature"].head(k).tolist()

    model = XGBRegressor(**XGB_PARAMS)
    model.fit(X_train[top_features], y_train)

    y_pred = model.predict(X_val[top_features])

    mse = mean_squared_error(y_val, y_pred)

    results.append({
        "method": "XGBoost Importance",
        "n_selected_features": k,
        "selected_features": top_features,
        "RMSE": mse ** 0.5,
        "MAE": mean_absolute_error(y_val, y_pred),
        "R2": r2_score(y_val, y_pred),
    })

xgb_results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
xgb_results_df


,method,n_selected_features,selected_features,RMSE,MAE,R2
0,Baseline (predict train mean),0,[],0.105132,0.077442,-0.008612
1,XGBoost Importance,5,"[sma_60, atr_14, sma_5, roc_20, volatility_20]",0.108197,0.082639,-0.068267
2,XGBoost Importance,10,"[sma_60, atr_14, sma_5, roc_20, volatility_20,...",0.109556,0.083282,-0.095268
3,XGBoost Importance,20,"[sma_60, atr_14, sma_5, roc_20, volatility_20,...",0.110325,0.083032,-0.110705
4,XGBoost Importance,15,"[sma_60, atr_14, sma_5, roc_20, volatility_20,...",0.113432,0.085628,-0.174141


## 3. Cross-check against RandomForest Importance's top features

Do the two non-linear models agree on which features matter?

In [5]:
rf_importance_path = PROJECT_ROOT / "data" / "processed" / "embedded_results" / "random_forest_importance_full.csv"

if rf_importance_path.exists():
    rf_importance_df = pd.read_csv(rf_importance_path)
    rf_top10 = set(rf_importance_df["feature"].head(10))
    xgb_top10 = set(importance_df["feature"].head(10))

    print("RF top10 ∩ XGB top10:", rf_top10 & xgb_top10)
    print("RF only:", rf_top10 - xgb_top10)
    print("XGB only:", xgb_top10 - rf_top10)
else:
    print("Run random_forest_importance.ipynb first to compare.")


RF top10 ∩ XGB top10: {'atr_14', 'macd_hist', 'volatility_20', 'sma_5', 'sma_20', 'sma_60'}
RF only: {'macd_signal', 'rsi_14', 'volume_sma_20', 'price_to_sma_60'}
XGB only: {'macd', 'return_20d', 'roc_20', 'price_to_sma_20'}


In [6]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "embedded_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

importance_df.to_csv(OUTPUT_DIR / "xgboost_importance_full.csv", index=False)
xgb_results_df.to_csv(OUTPUT_DIR / "xgboost_importance_results.csv", index=False)

print("Saved to:", OUTPUT_DIR)


Saved to: /Users/yangjaehoon/Desktop/StockLens/data/processed/embedded_results


## 4. Early stopping retry

The fixed `n_estimators=300` run above had no way to stop before
overfitting a noisy target. Here `n_estimators` is raised to a high
ceiling (1000) but training stops once validation RMSE hasn't improved
for 30 rounds, using `X_val`/`y_val` as the early-stopping monitor.

**Caveat**: this means the validation set now influences *when* training stops, not just the final reported score. That's a much softer use of validation than fitting hyperparameters to it, and is standard practice for boosting -- but it does mean these numbers carry slightly more optimism than the fixed-n_estimators run above. `test.csv` is still untouched.

In [7]:
XGB_ES_PARAMS = dict(
    n_estimators=1000,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror",
    early_stopping_rounds=30,
    eval_metric="rmse",
)

results_es = [baseline_mean_metrics(y_train, y_val)]

for k in top_k_list:
    top_features = importance_df["feature"].head(k).tolist()

    model = XGBRegressor(**XGB_ES_PARAMS)
    model.fit(
        X_train[top_features], y_train,
        eval_set=[(X_val[top_features], y_val)],
        verbose=False,
    )

    y_pred = model.predict(X_val[top_features])
    mse = mean_squared_error(y_val, y_pred)

    results_es.append({
        "method": "XGBoost Importance (early stopping)",
        "n_selected_features": k,
        "selected_features": top_features,
        "best_iteration": model.best_iteration,
        "RMSE": mse ** 0.5,
        "MAE": mean_absolute_error(y_val, y_pred),
        "R2": r2_score(y_val, y_pred),
    })

xgb_es_results_df = pd.DataFrame(results_es).sort_values("RMSE").reset_index(drop=True)
xgb_es_results_df


,method,n_selected_features,selected_features,RMSE,MAE,R2,best_iteration
0,XGBoost Importance (early stopping),5,"[sma_60, atr_14, sma_5, roc_20, volatility_20]",0.101902,0.076343,0.052417,40.0
1,XGBoost Importance (early stopping),10,"[sma_60, atr_14, sma_5, roc_20, volatility_20,...",0.101956,0.075795,0.051424,17.0
2,XGBoost Importance (early stopping),15,"[sma_60, atr_14, sma_5, roc_20, volatility_20,...",0.102717,0.075743,0.037208,12.0
3,XGBoost Importance (early stopping),20,"[sma_60, atr_14, sma_5, roc_20, volatility_20,...",0.103177,0.076644,0.028555,16.0
4,Baseline (predict train mean),0,[],0.105132,0.077442,-0.008612,NaN


In [8]:
xgb_es_results_df.to_csv(
    PROJECT_ROOT / "data" / "processed" / "embedded_results" / "xgboost_importance_early_stopping_results.csv",
    index=False,
)
print("Saved.")


Saved.
